# 03. Model Storage & Capacity Gate

Evaluates **authoritative Google Drive API account quota** before initiating the 142-shard model transfer into `/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model`.

### Capacity Semantics
| Layer | Source | Role |
|---|---|---|
| **Google Drive Account Quota** | `drive.about.get()` API | **Authoritative gate** — must have >= 400 GB free |
| **Colab Local NVMe** | `shutil.disk_usage('/content')` | Temp buffer only — 3 GiB per chunk, never full model |
| **FUSE Mount Capacity** | `shutil.disk_usage(drive_mount_path)` | **Diagnostic only** — never used as a gate |

> **The full 399.79 GiB model is NOT staged locally.** It downloads directly to Google Drive.

### Step 1: Pre-Download Capacity Gate

In [ ]:
import os
import sys
import shutil
import subprocess

# Ensure repository is present and synchronized to latest commit
REPO_DIR = '/content/glm52-drive-runtime'
if not os.path.exists(REPO_DIR):
    print('Cloning GLM-5.2 repository into ' + REPO_DIR + '...')
    subprocess.run(['git', 'clone', 'https://github.com/Aqib2607/AI.git', REPO_DIR], check=True)
else:
    print('Synchronizing ' + REPO_DIR + ' to latest master...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'master'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.drive_check import get_drive_service, get_drive_storage_quota, evaluate_storage_gate
from scripts.download_model import get_capacity_report, evaluate_download_gate, is_drive_path

# ─── Model Specifications ────────────────────────────────────────────────────
MODEL_REPO           = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'
MODEL_SIZE_GIB       = 399.79
MODEL_SIZE_GB        = 429.28
REQUIRED_DRIVE_GB    = 400.0
RECOMMENDED_DRIVE_GB = 450.0
TEMP_CHUNK_GIB       = 3.0      # Per-chunk local NVMe buffer only

# ─── Paths ───────────────────────────────────────────────────────────────────
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'

# ─── 1. Authoritative Google Drive API Quota ─────────────────────────────────
service  = get_drive_service()
capacity = get_capacity_report(DRIVE_MODEL_DIR, drive_service=service)

gate_status, gate_reason = evaluate_download_gate(
    capacity,
    required_gb=REQUIRED_DRIVE_GB,
    recommended_gb=RECOMMENDED_DRIVE_GB,
    local_temp_gib=TEMP_CHUNK_GIB
)

dq    = capacity['drive_api_quota']
local = capacity['local_colab_disk']
fuse  = capacity['fuse_diagnostic']

# ─── 2. Format values using intermediate variables (no nested f-strings) ─────
limit_text = ('{:,.2f} GB'.format(dq['limit_gb'])
              if dq.get('limit_gb') else 'Unlimited')
usage_text = ('{:,.2f} GB'.format(dq['usage_gb'])
              if dq.get('usage_gb') is not None else 'N/A')
if dq.get('free_gb') is not None:
    free_text = '{:,.2f} GB'.format(dq['free_gb'])
elif dq.get('is_unlimited'):
    free_text = 'Unlimited'
else:
    free_text = 'UNAVAILABLE'

fuse_free_text  = ('{:.2f} GiB'.format(fuse['free_gib'])
                   if fuse.get('free_gib') is not None else 'N/A')
fuse_total_text = ('{:.2f} GiB'.format(fuse['total_gib'])
                   if fuse.get('total_gib') is not None else 'N/A')

# ─── 3. Print Capacity Report ─────────────────────────────────────────────────
SEP = '=' * 75
sep = '-' * 75
print(SEP)
print('   GLM-5.2 COLIBRI  |  STORAGE CAPACITY PREFLIGHT REPORT')
print(SEP)
print('Model:                           ' + MODEL_REPO)
print('Model Size:                      {:.2f} GiB ({:.2f} GB decimal)'.format(MODEL_SIZE_GIB, MODEL_SIZE_GB))
print('Full model staged to local disk: NO - downloads directly to Google Drive')
print('Target path:                     ' + DRIVE_MODEL_DIR)
print('Target is Drive mount:           ' + str(is_drive_path(DRIVE_MODEL_DIR)))
print(sep)

print('SECTION 1 - Google Drive Account Quota (AUTHORITATIVE GATE)')
print('  Account:           ' + str(dq.get('email', 'aqibjawwad2607@gmail.com')))
print('  Total Plan Quota:  ' + limit_text)
print('  Used:              ' + usage_text)
print('  Available Free:    ' + free_text)
print('  Required:          >= {:.2f} GB  (Recommended: >= {:.2f} GB)'.format(REQUIRED_DRIVE_GB, RECOMMENDED_DRIVE_GB))
print('  GATE DECISION:     ' + gate_status)
print('  Reason:            ' + gate_reason)
print(sep)

print('SECTION 2 - Colab Local Ephemeral NVMe (TEMPORARY CHUNK BUFFER ONLY)')
print('  Local Free:        ' + str(local.get('free_gib', 'N/A')) + ' GiB')
print('  Local Total:       ' + str(local.get('total_gib', 'N/A')) + ' GiB')
print('  Required:          {:.2f} GiB (per-chunk temp only, not full model)'.format(TEMP_CHUNK_GIB))
print('  NOTE: The full {:.2f} GiB model is NOT copied to /content.'.format(MODEL_SIZE_GIB))
print(sep)

print('SECTION 3 - Drive FUSE Mount Diagnostics (INFORMATIONAL ONLY - NOT a gate)')
if fuse.get('free_gib') is not None:
    print('  FUSE Free:         ' + fuse_free_text + '  (Colab virtual container overlay)')
    print('  FUSE Total:        ' + fuse_total_text)
    print('  This value does NOT represent your Google Drive account storage capacity.')
else:
    print('  FUSE path not yet mounted or accessible.')
print(SEP)

# ─── 4. Final Decision ────────────────────────────────────────────────────────
# Success states: GO_WITH_RECOMMENDED_MARGIN (>= 500 GB free),
#                 GO (>= 450 GB free), GO_WITH_LOW_MARGIN (>= 400 GB free).
# Only NO-GO and NO-GO_UNKNOWN_QUOTA block the download.
is_go = gate_status in ('GO', 'GO_WITH_LOW_MARGIN', 'GO_WITH_RECOMMENDED_MARGIN')
decision_text = 'GO - READY FOR DOWNLOAD' if is_go else 'NO-GO - INSUFFICIENT GOOGLE DRIVE QUOTA'
print('PREFLIGHT DECISION: ' + decision_text)
print(SEP)

if not is_go:
    raise SystemExit('Download blocked: ' + gate_reason)
else:
    print('')
    print('Capacity gate passed. Proceed to Step 2 to start the download.')

### Step 2: Resumable Model Shard Download

Downloads directly to Google Drive. Preserves existing completed shards and resumes partial `.tmp` files automatically.

> **Authentication**: `mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp` is a public repository. An HF token is not required. If `HF_TOKEN` is already set in your Colab environment (Secrets or a prior cell), it will be used automatically — do not overwrite it here.

In [ ]:
import os

# The target repository is public - no token is required.
# If HF_TOKEN is already present in the environment (Colab Secrets, etc.),
# the downloader will pick it up automatically. Do not overwrite it here.

DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
MODEL_REPO      = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'

!python /content/glm52-drive-runtime/scripts/download_model.py \
  --repo "{MODEL_REPO}" \
  --target-dir "{DRIVE_MODEL_DIR}" \
  --required-gb 400 \
  --recommended-gb 450 \
  --local-temp-gib 3